In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(12345)

In [ ]:
class Material:
    def mu(self, E):
        raise NotImplementedError


class FlatMaterial(Material):
    def __init__(self, mu0):
        self.mu0 = mu0

    def mu(self, E):
        return np.full_like(E, self.mu0)


class ResonanceMaterial(Material):
    """
    Fake neutron absorption spectrum.

    Parameters
    ----------
    baseline : float
        Constant attenuation.
    peaks : list of tuples
        (centre, width, amplitude)
    """

    def __init__(self, baseline, peaks):
        self.baseline = baseline
        self.peaks = peaks

    def mu(self, E):

        mu = np.full_like(E, self.baseline)

        for centre, width, amp in self.peaks:
            mu += amp * np.exp(
                -0.5 * ((E - centre) / width) ** 2
            )

        return mu

In [ ]:
class Rectangle:

    def __init__(self, xmin, xmax, ymin, ymax, material):
        self.xmin = xmin
        self.xmax = xmax
        self.ymin = ymin
        self.ymax = ymax
        self.material = material

    def contains(self, x, y):
        return (
            (x >= self.xmin)
            & (x <= self.xmax)
            & (y >= self.ymin)
            & (y <= self.ymax)
        )

In [ ]:
background = FlatMaterial(0.9)

fake_fe = ResonanceMaterial(
    baseline=0.08,
    peaks=[
        (1.3, 0.05, 1.5),
        (2.4, 0.12, 0.8),
        (4.8, 0.20, 0.6),
        (6.5, 0.08, 1.2),
    ],
)

insert = Rectangle(
    35,
    65,
    40,
    60,
    fake_fe,
)

In [ ]:
E = np.linspace(0.5, 8.0, 1000)

plt.figure(figsize=(8,4))
plt.plot(E, background.mu(E), label="Slab")
plt.plot(E, fake_fe.mu(E), label="Insert")
plt.xlabel("Energy (eV)")
plt.ylabel(r"$\mu(E)$")
plt.legend()
plt.show()

In [ ]:
N = 10_000_000

x = rng.uniform(0, 100, N)
y = rng.uniform(0, 100, N)

# Uniform energy for now
E = rng.uniform(0.5, 8.0, N)

In [ ]:
inside = insert.contains(x, y)

mu = background.mu(E)

mu[inside] = insert.material.mu(E[inside])

thickness = 1.0

transmission = np.exp(-mu * thickness)

keep = rng.random(N) < transmission

In [ ]:
events = pd.DataFrame(
    {
        "x": x[keep],
        "y": y[keep],
        "E": E[keep],
    }
)

print(events.head())
print()
print(f"Detected events : {len(events):,}")

In [ ]:
plt.figure(figsize=(6,6))

plt.hist2d(
    events.x,
    events.y,
    bins=200,
)

plt.xlabel("x (mm)")
plt.ylabel("y (mm)")
plt.colorbar(label="Counts")

plt.show()

In [ ]:
inside_det = insert.contains(events.x.values,
                             events.y.values)

plt.figure(figsize=(8,4))

plt.hist(
    events.E[~inside_det],
    bins=120,
    density=True,
    histtype="step",
    label="Background",
)

plt.hist(
    events.E[inside_det],
    bins=120,
    density=True,
    histtype="step",
    label="Insert",
)

plt.xlabel("Energy (eV)")
plt.ylabel("Probability density")
plt.legend()

plt.show()

In [ ]:
from PIL import Image, ImageDraw

def make_meterial_patch(text: str, value: int) -> np.ndarray:
    img = Image.new("L", (64, 64), 0)

    draw = ImageDraw.Draw(img)
    for i in range(0, 64, 12):
        draw.text((0, i), (text + " ") * 15, fill=255)
    a = np.array(img)
    # return a, img
    # print(a.min(), a.max())
    return np.where(a > 0, value, 0).astype(int)

In [ ]:
# table = np.empty((9, 18), dtype=str)
# table[...] = ""
# table[0, 0] = "H"
# table[0, 17] = "He"
# table[3, 7] = "Fe"



# element coordinates in the periodic table
elements = {
    "H": {"loc": (0, 0), "material": ResonanceMaterial(
        baseline=0.08,
        peaks=[
            (1.3, 0.05, 1.5),
            (2.4, 0.12, 0.8),
            (4.8, 0.20, 0.6),
            (6.5, 0.08, 1.2),
        ],
    ), "value": 1},
    "He": {"loc": (0, 17), "material": ResonanceMaterial(
        baseline=0.08,
        peaks=[
            (3.3, 0.05, 1.5),
            (5.6, 0.12, 0.8),
            (6.5, 0.20, 0.6),
        ],
    ), "value": 2},
    # "Li": (1, 0),
    # "Be": (1, 1),
    # "B": (1, 12),
    # "C": (1, 13),
    # "N": (1, 14),
    # "O": (1, 15),
    # "F": (1, 16),
    # "Ne": (1, 17),
    # "Na": (2, 0),
    # "Mg": (2, 1),
    # "Al": (2, 12),
    # "Si": (2, 13),
    # "P": (2, 14),
    # "S": (2, 15),
    # "Cl": (2, 16),
    # "Ar": (2, 17),
    # "K": (3, 0),
    # "Ca": (3, 1),
    # "Sc": (3, 2),
    # "Ti": (3, 3),
    # "V": (3, 4),
    # "Cr": (3, 5),
    # "Mn": (3, 6),
    # "Fe": (3, 7),
    # "Co": (3, 8),
    # "Ni": (3, 9),
    # "Cu": (3, 10),
    # "Zn": (3, 11),
    # "Ga": (3, 12),
    # "Ge": (3, 13),
    # "As": (3, 14),
    # "Se": (3, 15),
    # "Br": (3, 16),
    # "Kr": (3, 17),
    # "Rb": (4, 0),
    # "Sr": (4, 1),
    # "Y": (4, 2),
    # "Zr": (4, 3),
    # "Nb": (4, 4),
    # "Mo": (4, 5),
    # "Tc": (4, 6),
    # "Ru": (4, 7),
    # "Rh": (4, 8),
    # "Pd": (4, 9),
    # "Ag": (4, 10),
    # "Cd": (4, 11),
    # "In": (4, 12),
    # "Sn": (4, 13),
    # "Sb": (4, 14),
    # "Te": (4, 15),
    # "I": (4, 16),
    # "Xe": (4, 17),
    # "Cs": (5, 0),
    # "Ba": (5, 1),
    # "La": (5, 2),
    # "Hf": (5, 3),
    # "Ta": (5, 4),
    # "W": (5, 5),
    # "Re": (5, 6),
    # "Os": (5, 7),
    # "Ir": (5, 8),
    # "Pt": (5, 9),
    # "Au": (5, 10),
    # "Hg": (5, 11),
    # "Tl": (5, 12),
    # "Pb": (5, 13),
    # "Bi": (5, 14),
    # "Po": (5, 15),
    # "At": (5, 16),
    # "Rn": (5, 17),
    # "Fr": (6, 0),
    # "Ra": (6, 1),
    # "Ac": (6, 2),
    # "Rf": (6, 3),
    # "Db": (6, 4),
    # "Sg": (6, 5),
    # "Bh": (6, 6),
    # "Hs": (6, 7),
    # "Mt": (6, 8),
    # "Ds": (6, 9),
    # "Rg": (6, 10),
    # "Cn": (6, 11),
    # "Nh": (6, 12),
    # "Fl": (6, 13),
    # "Mc": (6, 14),
    # "Lv": (6, 15),
    # "Ts": (6, 16),
    # "Og": (6, 17),
    # "Ce": (7, 3),
    # "Pr": (7, 4),
    # "Nd": (7, 5),
    # "Pm": (7, 6),
    # "Sm": (7, 7),
    # "Eu": (7, 8),
    # "Gd": (7, 9),
    # "Tb": (7, 10),
    # "Dy": (7, 11),
    # "Ho": (7, 12),
    # "Er": (7, 13),
    # "Tm": (7, 14),
    # "Yb": (7, 15),
    # "Lu": (7, 16),
    # "Th": (8, 3),
    # "Pa": (8, 4),
    # "U": (8, 5),
    # "Np": (8, 6),
    # "Pu": (8, 7),
    # "Am": (8, 8),
    # "Cm": (8, 9),
    # "Bk": (8, 10),
    # "Cf": (8, 11),
    # "Es": (8, 12),
    # "Fm": (8, 13),
    # "Md": (8, 14),
    # "No": (8, 15),
    # "Lr": (8, 16),
}

element_list = list(elements.keys())

In [ ]:
dx = 64
pad = 20

nrows = 9
ncols = 18

nx = ncols * (dx + pad) + pad
ny = nrows * (dx + pad) + pad

materials = np.zeros((ny, nx), dtype=int)

for i, (element, data) in enumerate(elements.items()):
    row, col = data["loc"]
    x0 = col * (dx + pad) + pad
    y0 = row * (dx + pad) + pad

    patch = make_meterial_patch(element, value=i + 1)

    materials[y0:y0+dx, x0:x0+dx] = patch

materials = np.flipud(materials)

In [ ]:
%matplotlib widget

In [ ]:
import plopp as pp

pp.plot(materials)

In [ ]:
from scipy.interpolate import interp1d

# 1. Pre-compute 1D transmission curves for each material
E_grid = np.linspace(0.5, 8.0, 200)
thickness = 1.0

# Background (material index 0)
transmission_curves = {
    0: np.exp(-background.mu(E_grid) * thickness)
}

# Each element
for i, (element, data) in enumerate(elements.items()):
    mat_idx = i + 1
    mu = data["material"].mu(E_grid)
    transmission_curves[mat_idx] = np.exp(-mu * thickness)

# 2. Create interpolators
interpolators = {
    idx: interp1d(E_grid, trans, bounds_error=False, fill_value=0)
    for idx, trans in transmission_curves.items()
}



# Generate events
N = 100_000_000

x = rng.uniform(0, materials.shape[1], N)
y = rng.uniform(0, materials.shape[0], N)

# Uniform energy for now
E = rng.uniform(0.5, 8.0, N)

# 3. Apply to events
# Look up material index for each event
mat_indices = materials[y.astype(int), x.astype(int)]


# Compute transmission for each event using its material's curve
transmission = np.zeros(N)
for mat_idx in np.unique(mat_indices):
    mask = mat_indices == mat_idx
    transmission[mask] = interpolators[mat_idx](E[mask])

keep = rng.random(N) < transmission

In [ ]:
import scipp as sc

In [ ]:
events = sc.DataArray(
    data=sc.ones(sizes={"event": np.count_nonzero(keep)}),
    coords={
        "x": sc.array(dims=["event"], values=x[keep], unit="mm"),
        "y": sc.array(dims=["event"], values=y[keep], unit="mm"),
        "E": sc.array(dims=["event"], values=E[keep], unit="keV"),
    }
)

print(f"Detected events : {len(events):,}")

In [ ]:
events.hist(y=776, x=1532).plot()

In [ ]:
materials.shape

In [ ]:
H = events.bin(x=sc.array(dims=["x"], values=[10., 90.], unit="mm"),
               y=sc.array(dims=["y"], values=[690., 760.], unit="mm"))
H

In [ ]:
H.squeeze().hist(E=300).plot()

In [ ]:
He = events.bin(x=sc.array(dims=["x"], values=[1440., 1515.], unit="mm"),
               y=sc.array(dims=["y"], values=[690., 760.], unit="mm"))
He.squeeze().hist(E=300).plot()

In [ ]:
class Realistic1vMaterial(Material):
    """
    Simplified but physically motivated neutron cross-section.

    Combines:
    - 1/v absorption (proportional to 1/sqrt(E))
    - Constant scattering
    - Optional resonance peaks (Breit-Wigner)
    """
    def __init__(self, sigma_abs_thermal, sigma_scatt, resonances=None):
        """
        Parameters
        ----------
        sigma_abs_thermal : float
            Absorption cross-section at 25 meV (barns)
        sigma_scatt : float
            Scattering cross-section (barns, approximately constant)
        resonances : list of tuples, optional
            Each tuple is (E_res_meV, width_meV, strength)
        """
        self.sigma_abs_thermal = sigma_abs_thermal
        self.sigma_scatt = sigma_scatt
        self.E_thermal = 25.0  # meV (thermal energy)
        self.resonances = resonances or []

    def mu(self, E_meV):
        """
        Return total cross-section in barns.

        Parameters
        ----------
        E_meV : array-like
            Neutron energy in meV
        """
        # 1/v absorption (goes as 1/sqrt(E))
        sigma_abs = self.sigma_abs_thermal * np.sqrt(self.E_thermal / E_meV)

        # Add resonances (Breit-Wigner formula)
        for E_res, width, strength in self.resonances:
            sigma_abs += strength * width**2 / ((E_meV - E_res)**2 + width**2)

        # Total cross-section
        sigma_total = sigma_abs + self.sigma_scatt

        # Convert barns to your units (adjust factor as needed)
        return sigma_total / 100.0

# Iron (Fe) - moderate absorber, small resonance around 1.15 keV (1150 meV)
fe_material = Realistic1vMaterial(
    sigma_abs_thermal=2.56,  # barns at 25 meV
    sigma_scatt=11.62,        # barns
    resonances=[
        (1150, 100, 5000),    # Resonance at ~1.15 eV
    ]
)

# Gold (Au) - strong absorber with famous 4.9 eV resonance!
au_material = Realistic1vMaterial(
    sigma_abs_thermal=98.65,  # barns at 25 meV (high!)
    sigma_scatt=9.3,           # barns
    resonances=[
        (4900, 150, 30000),   # Famous Au-197 resonance at 4.9 eV (4900 meV)
    ]
)

# Plot comparison over your energy range
energies_meV = np.logspace(np.log10(0.2), np.log10(10000), 500)  # 0.2 to 10000 meV

fe_xs = fe_material.mu(energies_meV) * 100  # Convert back to barns
au_xs = au_material.mu(energies_meV) * 100

plt.figure(figsize=(12, 6))
plt.loglog(energies_meV, fe_xs, label='Fe (Iron)', linewidth=2)
plt.loglog(energies_meV, au_xs, label='Au (Gold)', linewidth=2)
plt.axvline(25, color='gray', linestyle='--', alpha=0.5, label='Thermal (25 meV)')
plt.xlabel('Energy (meV)')
plt.ylabel('Total Cross-Section (barns)')
plt.title('Realistic Neutron Cross-Sections (0.2-10,000 meV)')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.xlim(0.2, 10000)
plt.show()

# Test at specific energies
test_E_meV = np.array([0.2, 1.0, 25, 100, 1000, 4900, 10000])
print("Energy (meV):", test_E_meV)
print("Fe xs (barns):", fe_material.mu(test_E_meV) * 100)
print("Au xs (barns):", au_material.mu(test_E_meV) * 100)

In [ ]:
import NCrystal as NC

# Load a material (can be element or compound)
# info = NC.createInfo("Al_sg225.ncmat")  # Aluminum
scatter = NC.createScatter("Al_sg225.ncmat")

# Get cross-section at specific energy
neutron_energy_eV = 25e-3  # 25 meV (thermal)
xs_total = scatter.xsect(ekin=neutron_energy_eV)  # in barns

# Or for a range of energies
energies = np.logspace(-3, 2, 1000)  # eV
xs_values = scatter.xsect(ekin=energies)

In [ ]:
pp.xyplot(energies, xs_values, logx=True, xlabel='Energy (eV)', ylabel='Cross-section (barns)', title='Aluminum Cross-Section from NCrystal',
          ls='solid', marker=None)

In [ ]:
scatter.plot()